In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()

    result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [2]:
llm= EasyLLM(provider="google_native")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)

2026-04-18 19:49:23,619 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash
2026-04-18 19:49:23,766 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: google_native


In [ ]:
test_invoke_without_tool(agent)

In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [ ]:
test_stream_without_tool(agent)

In [ ]:
await test_astream_without_tool(agent)

In [3]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-18 19:49:28,789 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-18 19:49:28,790 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-18 19:49:28,791 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-18 19:49:28,791 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [4]:
test_invoke_with_tool(agent)

2026-04-18 19:49:30,451 | INFO | 对话历史已清空
2026-04-18 19:49:30,452 | INFO | 使用工具模式调用智能体
2026-04-18 19:49:35,585 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1beta/models/gemini-3-flash:generateContent "HTTP/1.1 200 OK"
2026-04-18 19:49:35,587 | INFO | 思考内容: **Considering the Task's Scope**

I'm now zeroing in on the initial translation, and assessing its fidelity. After that, I will immediately start on the exponentiation. The response to the user should clearly and concisely deliver the information, along with a note on the translation accuracy.



2026-04-18 19:49:35,588 | WARNING | Warning: there are non-text parts in the response: ['function_call', 'function_call'], returning concatenated text result from text parts. Check the full candidates.content.parts accessor to get the full model response.
2026-04-18 19:49:35,588 | INFO | test_skill执行工具: translate_tool，参数: {'target_lang': 'en', 'text': '你是谁，在哪里'}
2026-04-18 19:49:35,589 | INFO | test_skill执行工具: calculator，参数: {'expre

翻译结果如下：

1.  **翻译结果**：
    *   原文：你是谁，在哪里
    *   翻译：Who are you and where are you?
    *   **判断**：翻译工具在本次执行中返回了原文（可能由于接口调用策略或识别问题），但在实际语境下，该句应翻译为 "Who are you and where are you?"。

2.  **计算结果**：
    *   $3^{22} = 31,381,059,609$


In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [5]:
await test_astream_with_tool(agent)

2026-04-18 19:50:14,324 | INFO | 对话历史已清空


round 1

thinking content:
**Analyzing User Intent**

I'm now focusing on the user's explicit requests. My current thinking is on how to best utilize the available tools to address the two distinct tasks. I need to ensure that the translation is accurate and the calculation is performed correctly, then verify the results.


**Executing the Plan**

I've just called the translation tool and reviewed the output. It seems spot-on. Next up, I'll move on to the calculation. Once the number crunching is complete, the final step involves neatly presenting both results and a quick verification of the translation's accuracy.



tool_calls:
translate_tool : {'text': '你是谁，在哪里', 'target_lang': 'en'}
calculator : {'expression': '3**22'}

round 2

content:
翻译结果如下：

1.  **翻译部分**：
    *   原文：你是谁，在哪里
    *   英语翻译：**Who are you, and where are you?**
    *   **翻译工具判断**：翻译工具在执行时返回的是原文，这通常是由于调用参数或处理逻辑的偏差。我已手动为您提供了准确的英语翻译。

2.  **计算部分**：
    *   $3^{22}$ 的计算结果为：**31,381,059,609**
final res:
翻译结果如下：

1.  **翻译

In [6]:
raw_history=agent.get_raw_history()  


In [7]:
llm= EasyLLM(provider="google",base_url="http://210.45.70.84:30000/v1")
agent.change_model(llm=llm)

2026-04-17 21:15:27,620 | INFO | EasyLLM 初始化完成: provider=google, model=gemini-3-flash


In [8]:
raw_history2=agent.get_raw_history()  

In [ ]:
raw_history2

In [ ]:

await agent.astream_invoke(f"我们刚才聊了什么")


In [10]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent.change_model(llm=llm)

2026-04-17 21:16:58,994 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b


In [ ]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

In [11]:
await agent.astream_invoke(f"我们刚才聊了什么")


2026-04-17 21:17:01,894 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


round 1

thinking content:
用户问我们刚才聊了什么，我需要回顾一下对话历史。

从对话历史来看：
1. 用户请求我：
   - 使用工具将"你是谁，在哪里"翻译成英语
   - 计算3^22

2. 我调用两个工具：
   - translate_tool 翻译中文到英文
   - calculator 计算3**22

3. 结果显示：
   - 翻译工具未能正确翻译，返回了原文
   - 计算器正确返回了31381059609

4. 我分析了翻译工具的问题，并给出了正确的翻译

5. 现在用户问我们刚才聊了什么

我应该简洁地总结刚才的对话内容。

content:


我们刚才讨论了：

1. **翻译任务**：尝试将中文"你是谁，在哪里"翻译成英语。工具返回了原文，未能成功翻译（正确应为 "Who are you, and where are you?"）。

2. **数学计算**：计算 $3^{22}$，结果是 **31,381,059,609**。
final res:


我们刚才讨论了：

1. **翻译任务**：尝试将中文"你是谁，在哪里"翻译成英语。工具返回了原文，未能成功翻译（正确应为 "Who are you, and where are you?"）。

2. **数学计算**：计算 $3^{22}$，结果是 **31,381,059,609**。


'\n\n我们刚才讨论了：\n\n1. **翻译任务**：尝试将中文"你是谁，在哪里"翻译成英语。工具返回了原文，未能成功翻译（正确应为 "Who are you, and where are you?"）。\n\n2. **数学计算**：计算 $3^{22}$，结果是 **31,381,059,609**。'